# DDT vs LightGBM: Ames Housing Benchmark Profiles

This notebook compares the performance of **DDT (Distributional Decision Tree)** against LightGBM, XGBoost, and CatBoost on the Ames Housing dataset, across various hardware and architectural profiles.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, TargetEncoder
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from ddt import DDTRegressor
import time
import pickle
from IPython.display import display, HTML

# --- Metrics Definition ---
def calc_cce(y_true, q_lower, q_upper, target_coverage):
    empirical_coverage = np.mean((y_true >= q_lower) & (y_true <= q_upper))
    return np.abs(target_coverage - empirical_coverage)

def calc_apiw(q_lower, q_upper):
    return np.mean(q_upper - q_lower)

def calc_cvar(y_true, q_threshold):
    breaches = y_true[y_true > q_threshold]
    return np.mean(breaches) if len(breaches) > 0 else 0

def calc_hei(infer_time_sec, n_samples, model_obj):
    latency_us = (infer_time_sec / n_samples) * 1e6
    size_kb = len(pickle.dumps(model_obj)) / 1024
    return latency_us * size_kb


In [ ]:
BENCHMARK_PROFILES = {
    "1_vs_1_theory": {
        "name": "1-vs-1 Representational Capacity (The Theoretical Duel)",
        "description": (
            "Evaluates the intrinsic capacity of a single tree structure to capture "
            "complex uncertainty (bimodality/heavy tails) without relying on sequential "
            "ensembling. Exposes the 'Flaw of Averages' head-on."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5,
            "min_samples_leaf": 50,
            "n_target_bins": 32,
            "n_feature_bins": 64,
        },
        "competitors_params": {
            "n_estimators": 1,
            "max_depth": 5,
            "num_leaves": 32,
        }
    },
    
    "edge_microkernel": {
        "name": "Edge AI & Micro-Kernel (The Silicon Test)",
        "description": (
            "Extremely constrained hardware simulation (IoT, Robotics, HFT). "
            "Guarantees the model and its ECDFs fit entirely within a 32 KB L1 Data Cache."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5,
            "min_samples_leaf": 100,
            "n_target_bins": 32,
            "n_feature_bins": 32,
        },
        "competitors_params": {
            "n_estimators": 3,
            "max_depth": 5,
            "num_leaves": 32,
        }
    },
    
    "production_ensemble": {
        "name": "Production Battle (Single-Tree vs. 100-Tree Boosting)",
        "description": (
            "Compares a single robust DDT against standard heavy 100-tree boosting ensembles."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 8,
            "min_samples_leaf": 200,
            "n_target_bins": 64,
            "n_feature_bins": 64,
        },
        "competitors_params": {
            "n_estimators": 100,
            "max_depth": 8,
            "learning_rate": 0.05,
        }
    },
    
    "deep_resolution": {
        "name": "Deep Resolution Explorer (The Statistical Microscope)",
        "description": (
            "High-resolution probabilistic mapping for massive datasets."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 12,
            "min_samples_leaf": 500,
            "n_target_bins": 100,
            "n_feature_bins": 64,
        },
        "competitors_params": {
            "n_estimators": 100,
            "max_depth": 12,
            "learning_rate": 0.05,
        }
    }
}


In [ ]:
# --- DATA LOADING & PREPROCESSING ---
MAX_SAMPLES = None # Set to an integer like 100000 to subsample

ames = fetch_openml(name='house_prices', as_frame=True, parser='auto')
X = ames.data.copy()
y = ames.target.values.astype(np.float32)

# Clean target NaNs if any exist
non_nan_idx = ~np.isnan(y)
X = X[non_nan_idx]
y = y[non_nan_idx]

if MAX_SAMPLES is not None and len(X) > MAX_SAMPLES:
    np.random.seed(42)
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X.iloc[idx].reset_index(drop=True)
    y = y[idx]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def preprocess_pipeline(X_train_df, X_test_df, y_train_arr, target_strategy="standard", feature_scaling="standard"):
    """
    Executes a clean, leakage-free preprocessing pipeline.
    Winsorization and standard scaling can be applied to the TARGET.
    Features can be scaled using 'standard', 'minmax', or 'none'.
    """
    X_tr = X_train_df.copy()
    X_te = X_test_df.copy()
    y_tr_processed = y_train_arr.copy()
    
    # Target Encoding for high-cardinality categoricals (Origin, Dest, Carrier)
    if len(categorical_cols) > 0:
        encoder = TargetEncoder(smooth="auto", cv=5, random_state=42, target_type="continuous")
        X_tr_cat = encoder.fit_transform(X_tr[categorical_cols], y_train_arr)
        X_te_cat = encoder.transform(X_te[categorical_cols])
        
        # Merge back with numerical features
        X_tr_num = X_tr[numerical_cols].fillna(0).values
        X_te_num = X_te[numerical_cols].fillna(0).values
        
        X_train_out = np.hstack([X_tr_num, X_tr_cat])
        X_test_out = np.hstack([X_te_num, X_te_cat])
    else:
        X_train_out = X_tr[numerical_cols].fillna(0).values
        X_test_out = X_te[numerical_cols].fillna(0).values

    # Apply scaling strategies to FEATURES
    if feature_scaling == "standard":
        f_scaler = StandardScaler()
        X_train_out = f_scaler.fit_transform(X_train_out)
        X_test_out = f_scaler.transform(X_test_out)
    elif feature_scaling == "minmax":
        f_scaler = MinMaxScaler()
        X_train_out = f_scaler.fit_transform(X_train_out)
        X_test_out = f_scaler.transform(X_test_out)
    elif feature_scaling == "none":
        pass
    else:
        raise ValueError(f"Unknown feature scaling: {feature_scaling}")

    # Apply scaling strategies to TARGET
    y_scaler = None
    
    if target_strategy == "winsorized":
        lower_y = np.percentile(y_tr_processed, 1)
        upper_y = np.percentile(y_tr_processed, 99)
        y_tr_processed = np.clip(y_tr_processed, lower_y, upper_y)
        
    if target_strategy in ["standard", "winsorized"]:
        y_scaler = StandardScaler()
        y_tr_processed = y_scaler.fit_transform(y_tr_processed.reshape(-1, 1)).flatten()
        
    return X_train_out, X_test_out, y_tr_processed, y_scaler


In [ ]:
def run_benchmark(profile_name, profile, X_train_scaled, X_test_scaled, y_train, y_test, preproc_name, y_scaler=None):
    print(f"\n{'='*80}\nRunning Profile: {profile['name']} | Preprocessing: {preproc_name}\n{'-'*80}")
    
    n_test = len(y_test)
    ddt_p = profile['ddt_params'].copy()
    comp_p = profile['competitors_params'].copy()
    
    # -- Train DDT --
    ddt = DDTRegressor(**ddt_p)
    start_time = time.time()
    ddt.fit(X_train_scaled, y_train)
    ddt_fit_time = time.time() - start_time
    
    # DDT Predict
    start_time = time.time()
    ddt_preds = ddt.predict_quantiles(X_test_scaled, profile['target_quantiles'])
    ddt_infer_time = time.time() - start_time
    ddt_q05 = ddt_preds[profile['target_quantiles'][0]]
    ddt_q50 = ddt_preds[profile['target_quantiles'][1]]
    ddt_q95 = ddt_preds[profile['target_quantiles'][2]]

    # -- Train DDT-Log --
    ddt_log_p = ddt_p.copy()
    ddt_log_p["bin_strategy"] = "log_width"
    ddt_log_p["winsorize_tails"] = 0.001
    offset = abs(np.min(y_train)) + 1.0 if np.min(y_train) <= 0 else 0.0
    y_train_log = y_train + offset
    
    ddt_log = DDTRegressor(**ddt_log_p)
    start_time = time.time()
    ddt_log.fit(X_train_scaled, y_train_log)
    ddt_log_fit_time = time.time() - start_time
    
    start_time = time.time()
    ddt_log_preds = ddt_log.predict_quantiles(X_test_scaled, profile['target_quantiles'])
    ddt_log_infer_time = time.time() - start_time
    ddt_log_q05 = ddt_log_preds[profile['target_quantiles'][0]] - offset
    ddt_log_q50 = ddt_log_preds[profile['target_quantiles'][1]] - offset
    ddt_log_q95 = ddt_log_preds[profile['target_quantiles'][2]] - offset

    # -- Train DDT-Hybrid --
    ddt_hyb_p = ddt_p.copy()
    ddt_hyb_p["bin_strategy"] = "hybrid"
    ddt_hyb_p["core_fraction"] = 0.5
    
    ddt_hyb = DDTRegressor(**ddt_hyb_p)
    start_time = time.time()
    ddt_hyb.fit(X_train_scaled, y_train)
    ddt_hyb_fit_time = time.time() - start_time
    
    start_time = time.time()
    ddt_hyb_preds = ddt_hyb.predict_quantiles(X_test_scaled, profile['target_quantiles'])
    ddt_hyb_infer_time = time.time() - start_time
    ddt_hyb_q05 = ddt_hyb_preds[profile['target_quantiles'][0]]
    ddt_hyb_q50 = ddt_hyb_preds[profile['target_quantiles'][1]]
    ddt_hyb_q95 = ddt_hyb_preds[profile['target_quantiles'][2]]
    
    # -- Train LightGBM --
    lgbms = []
    start_time = time.time()
    for q in profile['target_quantiles']:
        lgbm = LGBMRegressor(objective='quantile', alpha=q, random_state=42, verbose=-1, **comp_p)
        lgbm.fit(X_train_scaled, y_train)
        lgbms.append(lgbm)
    lgbm_fit_time = time.time() - start_time
    
    start_time = time.time()
    lgbm_q05 = lgbms[0].predict(X_test_scaled)
    lgbm_q50 = lgbms[1].predict(X_test_scaled)
    lgbm_q95 = lgbms[2].predict(X_test_scaled)
    lgbm_infer_time = time.time() - start_time
    
    # -- Train XGBoost --
    xgbs = []
    start_time = time.time()
    for q in profile['target_quantiles']:
        xgb_p = comp_p.copy()
        if 'num_leaves' in xgb_p:
            # XGBoost uses max_leaves combined with lossguide to mimic num_leaves
            xgb_p['max_leaves'] = xgb_p.pop('num_leaves')
            xgb_p['grow_policy'] = 'lossguide'
        xgb = XGBRegressor(objective='reg:quantileerror', quantile_alpha=q, random_state=42, **xgb_p)
        xgb.fit(X_train_scaled, y_train)
        xgbs.append(xgb)
    xgb_fit_time = time.time() - start_time
    
    start_time = time.time()
    xgb_q05 = xgbs[0].predict(X_test_scaled)
    xgb_q50 = xgbs[1].predict(X_test_scaled)
    xgb_q95 = xgbs[2].predict(X_test_scaled)
    xgb_infer_time = time.time() - start_time
    
    # -- Train CatBoost --
    quantiles_str = ','.join([str(q) for q in profile['target_quantiles']])
    cat_p = comp_p.copy()
    if 'n_estimators' in cat_p:
        cat_p['iterations'] = cat_p.pop('n_estimators')
    if 'num_leaves' in cat_p:
        cat_p.pop('num_leaves') # CatBoost uses depth
    cat = CatBoostRegressor(loss_function=f'MultiQuantile:alpha={quantiles_str}', random_seed=42, verbose=False, **cat_p)
    
    start_time = time.time()
    cat.fit(X_train_scaled, y_train)
    cat_fit_time = time.time() - start_time
    
    start_time = time.time()
    cat_preds = cat.predict(X_test_scaled)
    cat_infer_time = time.time() - start_time
    cat_q05 = cat_preds[:, 0]
    cat_q50 = cat_preds[:, 1]
    cat_q95 = cat_preds[:, 2]
    
    # -- Inverse Transform Predictions --
    if y_scaler is not None:
        ddt_q05 = y_scaler.inverse_transform(ddt_q05.reshape(-1, 1)).flatten()
        ddt_q50 = y_scaler.inverse_transform(ddt_q50.reshape(-1, 1)).flatten()
        ddt_q95 = y_scaler.inverse_transform(ddt_q95.reshape(-1, 1)).flatten()
        
        lgbm_q05 = y_scaler.inverse_transform(lgbm_q05.reshape(-1, 1)).flatten()
        lgbm_q50 = y_scaler.inverse_transform(lgbm_q50.reshape(-1, 1)).flatten()
        lgbm_q95 = y_scaler.inverse_transform(lgbm_q95.reshape(-1, 1)).flatten()
        
        xgb_q05 = y_scaler.inverse_transform(xgb_q05.reshape(-1, 1)).flatten()
        xgb_q50 = y_scaler.inverse_transform(xgb_q50.reshape(-1, 1)).flatten()
        xgb_q95 = y_scaler.inverse_transform(xgb_q95.reshape(-1, 1)).flatten()
        
        cat_q05 = y_scaler.inverse_transform(cat_q05.reshape(-1, 1)).flatten()
        cat_q50 = y_scaler.inverse_transform(cat_q50.reshape(-1, 1)).flatten()
        cat_q95 = y_scaler.inverse_transform(cat_q95.reshape(-1, 1)).flatten()
    
    # -- Compute Metrics --
    target_cov = profile['target_quantiles'][2] - profile['target_quantiles'][0]
    
    results = {
        'Metric': [f'CCE ({int(target_cov*100)}% CI)', f'APIW ({int(target_cov*100)}% CI)', f'Realized CVaR (Tail > P{int(profile["target_quantiles"][2]*100)})', 'Train Time (s)', 'Latency (us/sample)', 'Model Size (KB)', 'HEI'],
        'DDT': [
            f"{calc_cce(y_test, ddt_q05, ddt_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(ddt_q05, ddt_q95):,.2f}",
            f"{calc_cvar(y_test, ddt_q95):,.2f}", 
            f"{ddt_fit_time:.2f}",
            f"{(ddt_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(ddt)) / 1024:.2f}",
            f"{calc_hei(ddt_infer_time, n_test, ddt):.2f}"
        ],
        'DDT-Log': [
            f"{calc_cce(y_test, ddt_log_q05, ddt_log_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(ddt_log_q05, ddt_log_q95):,.2f}",
            f"{calc_cvar(y_test, ddt_log_q95):,.2f}", 
            f"{ddt_log_fit_time:.2f}",
            f"{(ddt_log_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(ddt_log)) / 1024:.2f}",
            f"{calc_hei(ddt_log_infer_time, n_test, ddt_log):.2f}"
        ],
        'DDT-Hybrid': [
            f"{calc_cce(y_test, ddt_hyb_q05, ddt_hyb_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(ddt_hyb_q05, ddt_hyb_q95):,.2f}",
            f"{calc_cvar(y_test, ddt_hyb_q95):,.2f}", 
            f"{ddt_hyb_fit_time:.2f}",
            f"{(ddt_hyb_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(ddt_hyb)) / 1024:.2f}",
            f"{calc_hei(ddt_hyb_infer_time, n_test, ddt_hyb):.2f}"
        ],
        'LightGBM': [
            f"{calc_cce(y_test, lgbm_q05, lgbm_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(lgbm_q05, lgbm_q95):,.2f}",
            f"{calc_cvar(y_test, lgbm_q95):,.2f}",
            f"{lgbm_fit_time:.2f}",
            f"{(lgbm_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(lgbms)) / 1024:.2f}",
            f"{calc_hei(lgbm_infer_time, n_test, lgbms):.2f}"
        ],
        'XGBoost': [
            f"{calc_cce(y_test, xgb_q05, xgb_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(xgb_q05, xgb_q95):,.2f}",
            f"{calc_cvar(y_test, xgb_q95):,.2f}",
            f"{xgb_fit_time:.2f}",
            f"{(xgb_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(xgbs)) / 1024:.2f}",
            f"{calc_hei(xgb_infer_time, n_test, xgbs):.2f}"
        ],
        'CatBoost': [
            f"{calc_cce(y_test, cat_q05, cat_q95, target_cov)*100:.2f}%",
            f"{calc_apiw(cat_q05, cat_q95):,.2f}",
            f"{calc_cvar(y_test, cat_q95):,.2f}",
            f"{cat_fit_time:.2f}",
            f"{(cat_infer_time / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(cat)) / 1024:.2f}",
            f"{calc_hei(cat_infer_time, n_test, cat):.2f}"
        ]
    }
    
    df_results = pd.DataFrame(results).set_index('Metric')
    display(HTML(f"<h4>Results for {profile['name']} ({preproc_name})</h4>"))
    display(df_results)
    
    # Return formatted string for final accumulation
    res_str = f"=== Profile: {profile['name']} | Preprocessing: {preproc_name} ===\n"
    res_str += df_results.to_string() + "\n\n"
    return res_str


In [ ]:
all_results_text = ""

# Fixed feature scaling
FEATURE_SCALING = "standard"

for preproc in ["none", "standard", "winsorized"]:
    X_tr_s, X_te_s, y_tr_s, y_scaler = preprocess_pipeline(
        X_train_raw, X_test_raw, y_train, 
        target_strategy=preproc, 
        feature_scaling=FEATURE_SCALING
    )
    
    for prof_key, prof in BENCHMARK_PROFILES.items():
        res_text = run_benchmark(prof_key, prof, X_tr_s, X_te_s, y_tr_s, y_test, preproc, y_scaler=y_scaler)
        all_results_text += res_text


In [ ]:
print("--- FINAL COMPILED RESULTS ---")
print(all_results_text)
